# Tracking Cohort Data Preparation

This notebook builds MDS-UPDRS Parts I, II, and III tables for the
"Tracking" cohort, reshaped and renamed to match the PPMI MDS-UPDRS
column conventions — the same pattern used in `03_OPDC_Data_Prep.ipynb`
for the OPDC cohort.

**What this notebook does:**
1. Loads the Tracking longitudinal dataset and attaches subgroup labels.
2. Derives a `time_since_first_visit` field (years since each subject's
   first visit) to use in place of PPMI's visit-code `EVENT_ID`.
3. Recodes the severity-rating columns (`Normal`/`Slight`/`Mild`/
   `Moderate`/`Severe`) to PPMI-style ordinal scores (0-4).
4. Looks up the PPMI MDS-UPDRS column names and renames the matching
   Tracking columns to line up with them.
5. Splits each part into ON- and OFF-state tables (using the recorded
   `patients_clinical_state` field directly, unlike the OPDC notebook
   which derives ON/OFF from time-since-last-dose) and pickles them.



In [1]:
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns

### Load Tracking Cohort and Subgroup Labels

In [2]:
tracking = pd.read_csv('../../data/01_raw/TRACKING/P3_Tracking_Longitudinal_2025.csv')

subgroups = pd.read_csv('./../../data/01_raw/TRACKING/subgroups_raw.csv')

In [3]:
# Attach each participant's subgroup label. IDs are matched directly
# ('ID' <-> 'participant_id'), no site-prefix cleanup needed here (unlike
# OPDC's 'site/subjectnumber' format).
tracking = pd.merge(tracking, subgroups, left_on='ID', right_on='participant_id')

### Derive Time-Since-First-Visit

Same approach as the OPDC notebook: build a continuous time axis (years since each subject's first visit) to stand in for PPMI's `EVENT_ID`, since Tracking doesn't use PPMI-style visit codes.

In [4]:
tracking = tracking.sort_values(["ID", "Visit"]).reset_index(drop=True)

# Years elapsed since the previous visit for this subject (NaN on each
# subject's first visit).
tracking["visit_interval_years"] = (tracking.groupby("ID")["age_at_visit"].diff())

# Cumulative years since the subject's first visit.
tracking["time_since_first_visit"] = (
    tracking.groupby("ID")["visit_interval_years"].transform(lambda x: x.fillna(0).cumsum())
)

### Section 1 — Locate the MDS-UPDRS Columns

Unlike the OPDC notebook (which selects columns by position), Tracking's MDS-UPDRS columns are identified by name — any column containing `'UPDRS'`.

In [5]:
mds_updrs_cols = []

for col in tracking.columns:
    if "UPDRS" in col:
        mds_updrs_cols.append(col)

In [6]:
mds_updrs = tracking.loc[:, ['ID', 'time_since_first_visit', 'patients_clinical_state', 'hoehn_and_yahr_stage'] + mds_updrs_cols]

### Load PPMI Column Names for Renaming

Same as the OPDC notebook: read the PPMI source files far enough to grab their column names, which the matching Tracking columns are renamed to below.

In [7]:
ppmi_mds_updrs_p1_1 = pd.read_csv('./../../data/01_raw/PPMI/Assesment_Exams/MDS-UPDRS_Part_I_21Jan2025.csv')
ppmi_mds_updrs_p1_1_cols = [col for col in ppmi_mds_updrs_p1_1.columns if 'NP1' in col]

ppmi_mds_updrs_p1_2 = pd.read_csv('./../../data/01_raw/PPMI/Assesment_Exams/MDS-UPDRS_Part_I_Patient_Questionnaire_21Jan2025.csv')
ppmi_mds_updrs_p1_2_cols = [col for col in ppmi_mds_updrs_p1_2.columns if 'NP1' in col]

ppmi_mds_updrs_p2 = pd.read_csv('./../../data/01_raw/PPMI/Assesment_Exams/MDS_UPDRS_Part_II__Patient_Questionnaire_21Jan2025.csv')
ppmi_mds_updrs_p2_cols = [col for col in ppmi_mds_updrs_p2.columns if 'NP2' in col]

ppmi_mds_updrs_p3 = pd.read_csv('./../../data/01_raw/PPMI/Assesment_Exams/MDS-UPDRS_Part_III_21Jan2025.csv')
ppmi_mds_updrs_p3_cols = [col for col in ppmi_mds_updrs_p3.columns if 'NP3' in col] + ['patients_clinical_state', 'hoehn_and_yahr_stage']

ppmi_mds_updrs_p4 = pd.read_csv('./../../data/01_raw/PPMI/Assesment_Exams/MDS-UPDRS_Part_IV__Motor_Complications_21Jan2025.csv')
ppmi_mds_updrs_p4_cols = [col for col in ppmi_mds_updrs_p4.columns if 'NP4' in col]

### Recode Severity Ratings to Ordinal Scores

Tracking's item-level MDS-UPDRS columns are recorded as text severity labels rather than PPMI's 0-4 integer scale. This recodes every item column (skipping any `*total*` columns, which are already numeric, and the four ID/metadata columns) onto the ordered scale `Normal < Slight < Mild < Moderate < Severe`, then converts the category to its integer code (0-4). Values that don't match one of the five labels become `-1` from `.cat.codes` and are converted to `NaN`.

In [8]:
for col in mds_updrs.columns:
    if 'total' in col:
        continue
    elif col == 'ID' or col == 'time_since_first_visit' or col == 'patients_clinical_state' or col == 'hoehn_and_yahr_stage':
        continue
    else:
        mds_updrs[col] = pd.Categorical(
            mds_updrs[col],
            ordered=True,
            categories=['Normal', 'Slight', 'Mild', 'Moderate', 'Severe']
        )
        mds_updrs[col] = mds_updrs[col].cat.codes
        mds_updrs[col] = mds_updrs[col].replace(-1, np.nan)

### Build Part I, II, III Tables

For each part: select the Tracking columns belonging to that part (by `'_I_'` / `'_II_'` / `'_III_'` substring), rename them to the matching PPMI item codes, and carry along `State` (`patients_clinical_state`) for the ON/OFF split below.


In [9]:
p1_final = mds_updrs[['ID', 'time_since_first_visit', 'patients_clinical_state'] + [col for col in mds_updrs.columns if '_I_' in col]]
p1_final.columns = ['ID', 'time_since_first_visit', 'patients_clinical_state'] + ppmi_mds_updrs_p1_1_cols[:-1] + ppmi_mds_updrs_p1_2_cols[:-1] + ['NP1TOT']
p1_final = p1_final.rename(columns={'ID': 'PATNO', 'time_since_first_visit': 'EVENT_ID', 'patients_clinical_state': 'State'})

p2_final = mds_updrs[['ID', 'time_since_first_visit', 'patients_clinical_state'] + [col for col in mds_updrs.columns if '_II_' in col]]
p2_final.columns = ['ID', 'time_since_first_visit', 'patients_clinical_state'] + ppmi_mds_updrs_p2_cols
p2_final = p2_final.rename(columns={'ID': 'PATNO', 'time_since_first_visit': 'EVENT_ID', 'patients_clinical_state': 'State'})

p3_final = mds_updrs[['ID', 'time_since_first_visit', 'patients_clinical_state', 'hoehn_and_yahr_stage'] + [col for col in mds_updrs.columns if '_III_' in col]]
p3_final.columns = ['ID', 'time_since_first_visit', 'patients_clinical_state', 'hoehn_and_yahr_stage'] + ppmi_mds_updrs_p3_cols[:-2]
p3_final = p3_final.rename(columns={'ID': 'PATNO', 'time_since_first_visit': 'EVENT_ID', 'patients_clinical_state': 'State', 'hoehn_and_yahr_stage': 'NHY'})

### Split Each Part by Medication State and Save

Splits on the literal values `'On'` / `'Off'` from `patients_clinical_state` (note the capitalization — different from the `'ON'`/`'OFF'` used in the OPDC notebook, which derives its state flag rather than reading it directly).

#### Part I

In [10]:
p1_final_tmp = p1_final.copy()

p1_final = p1_final_tmp[p1_final_tmp['State'].isin([np.nan , 'On'])].drop(columns=['State'])
with open('./../../data/02_processed/TRACKING/P1ON_MDSUPDRS.pkl', 'wb') as file:
    pickle.dump(p1_final, file)

p1_final = p1_final_tmp[p1_final_tmp['State'].isin([np.nan , 'Off'])].drop(columns=['State'])
with open('./../../data/02_processed/TRACKING/P1OFF_MDSUPDRS.pkl', 'wb') as file:
    pickle.dump(p1_final, file)

#### Part II

In [11]:
p2_final_tmp = p2_final.copy()

p2_final = p2_final_tmp[p2_final_tmp['State'].isin([np.nan , 'On'])].drop(columns=['State'])
with open('./../../data/02_processed/TRACKING/P2ON_MDSUPDRS.pkl', 'wb') as file:
    pickle.dump(p2_final, file)

p2_final = p2_final_tmp[p2_final_tmp['State'].isin([np.nan , 'Off'])].drop(columns=['State'])
with open('./../../data/02_processed/TRACKING/P2OFF_MDSUPDRS.pkl', 'wb') as file:
    pickle.dump(p2_final, file)

#### Part III

In [12]:
p3_final_tmp = p3_final.copy()

p3_final = p3_final_tmp[p3_final_tmp['State'].isin([np.nan , 'On'])].drop(columns=['State'])
with open('./../../data/02_processed/TRACKING/P3ON_MDSUPDRS.pkl', 'wb') as file:
    pickle.dump(p3_final, file)

p3_final = p3_final_tmp[p3_final_tmp['State'].isin([np.nan , 'Off'])].drop(columns=['State'])
with open('./../../data/02_processed/TRACKING/P3OFF_MDSUPDRS.pkl', 'wb') as file:
    pickle.dump(p3_final, file)